# Implement trained model using HLS4ML
Having downloaded the dataset, defined the CNN model and trained with pertinent data, it's time to synthesize it using HLS4ML.

In [7]:
import os, json, re, warnings
import tensorflow as tf
import pickle

os.environ['XILINX_VITIS'] = '/tools/Xilinx/Vitis/2024.1'
os.environ['XILINX_HLS'] = '/tools/Xilinx/Vitis_HLS/2024.1'
os.environ['PATH'] += ':/tools/Xilinx/Vitis_HLS/2024.1/bin'
os.environ['PATH'] = os.environ['XILINX_VITIS'] + '/bin:' + os.environ['PATH']


# QKeras – quantization-aware training
from qkeras import QConv2D, QDense, QActivation, quantized_bits, quantized_relu
from qkeras.utils import _add_supported_quantized_objects

from tensorflow.keras.models import Sequential
from tensorflow.keras.models import load_model

from tensorflow.keras.layers import (InputLayer, 
                                     BatchNormalization,
                                     MaxPooling2D,
                                     Flatten,
                                     Dropout                                
)
from tensorflow.keras.optimizers import Adam

co = {}
_add_supported_quantized_objects(co)
model_MFCC = load_model('CNN_Model_Trained/Trained_model_100e.h5', custom_objects=co)
# hls4ml – HLS model conversion and synthesis
import hls4ml

## Reproducibility
RANDOM_SEED = 55
#np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
warnings.filterwarnings('ignore')


# ── FPGA target (Kria KV260-compatible) ───────────────────────────────────
FPGA_PART    = 'xck26-sfvc784-2LV-c'
CLOCK_PERIOD = 5   # ns → 200 MHz
TARGET_SNR   = '76dB'

# ── hls4ml output ─────────────────────────────────────────────────────────
HLS_BASE_DIR = './hls4ml_output'

# ── Training hyper-parameters ─────────────────────────────────────────────
BATCH_SIZE    = 32
MAX_EPOCHS    = 100
LEARNING_RATE = 1e-4
PATIENCE      = 20



# Load dataset parameters for QAT

In [8]:
# ── Datasets parameters ─────────────────────────────────────────────
DATASETS_PATH = "MFCC_datasets"
METADATA_PATH = os.path.join(DATASETS_PATH, "metadata.json")

if os.path.exists(METADATA_PATH):
    with open(METADATA_PATH, "r") as f:
        metadata = json.load(f)
    MFCC_input_shape = tuple(metadata["MFCC_input_shape"])
    NUM_CLASSES      = metadata["num_classes"]
    N_MFCC           = metadata.get("N_MFCC", 20)
    MFCC_FRAMES      = metadata.get("MFCC_FRAMES")
    SAMPLING_FREQ    = metadata.get("SAMPLING_FREQ", 48000)
    batch_size       = metadata.get("batch_size", 32)
    subfolders       = metadata.get("subfolders", [])
    unique_labels    = metadata.get("labels", [])
    print("Metadata loaded from metadata.json")
else:
    # Fallback: infer parameters from the dataset after loading
    print("Warning: metadata.json not found. Parameters will be inferred from the dataset.")
    N_MFCC        = 20
    SAMPLING_FREQ = 48000
    batch_size    = 32
    subfolders    = []
    MFCC_input_shape = None
    NUM_CLASSES      = None

print(f"  MFCC input shape : {MFCC_input_shape}")
print(f"  Num classes      : {NUM_CLASSES}")
print(f"  N_MFCC           : {N_MFCC}")
print(f"  MFCC_FRAMES      : {MFCC_FRAMES}")
print(f"  Sampling freq    : {SAMPLING_FREQ} Hz")
print(f"  Batch size       : {batch_size}")
print(f"  Classes          : {subfolders}")


Metadata loaded from metadata.json
  MFCC input shape : (20, 282, 2)
  Num classes      : 10
  N_MFCC           : 20
  MFCC_FRAMES      : 282
  Sampling freq    : 48000 Hz
  Batch size       : 32
  Classes          : ['f0001', 'f0002', 'f0003', 'f0004', 'f0005', 'm0001', 'm0002', 'm0003', 'm0004', 'm0005']


# Quantization-Aware Training (QAT)

Three QKeras models are fine-tuned starting from the float weights:

| Model | Weight / activation precision |
|---|---|
| QAT 16-bit | `quantized_bits(16, 6)` |
| QAT 8-bit  | `quantized_bits(8, 4)`  |
| QAT 4-bit  | `quantized_bits(4, 2)`  |

Fine-tuning starts from the float checkpoint; fewer epochs and a lower
learning rate let the network adapt to reduced precision without
catastrophic forgetting.

In [16]:
MFCC_input_shape =  model_MFCC.input_shape

def build_qat_model(input_shape, n_classes, bits, integer_bits):

    #Quantizers
    kq = quantized_bits(bits, integer_bits, keep_negative=1, alpha=1)
    aq = quantized_relu(bits, integer_bits)

    q_Model = Sequential([
        InputLayer(shape=MFCC_input_shape),

        #Block 1
        QConv2D(32, (3,3), padding='same',
                kernel_quantizer=kq, bias_quantizer=kq, name='conv1'),
        BatchNormalization(name='bn1'),
        QActivation(activation=aq, name='act1'),
        MaxPooling2D((2,2), name='pool1'),

        #Block 2
        QConv2D(64, (3,3), padding='same',
                kernel_quantizer=kq, bias_quantizer=kq, name='conv2'),
        BatchNormalization(name='bn2'),
        QActivation(activation=aq, name='act2'),
        MaxPooling2D((2,2), name='pool2'),

        #Classifier
        Flatten(name='flatten'),
        QDense(128, kernel_quantizer=kq, bias_quantizer=kq, name='dense1'),
        QActivation(activation=aq, name='act3'),
        Dropout(0.5, name='dropout'),

        QDense(n_classes, kernel_quantizer=kq, bias_quantizer=kq, name='output'),
        
        #Softmax remains in float for stability.
        tf.keras.layers.Activation('softmax', name='softmax'),
    ], name=f'MFCC_CNN_QAT_{bits}bit')

    return q_Model


def transfer_float_weights(float_model, qat_model):
    float_wl = [l for l in float_model.layers if l.get_weights()]
    qat_wl   = [l for l in qat_model.layers  if l.get_weights()]
    n_ok = 0
    for fl, ql in zip(float_wl, qat_wl):
        try:
            ql.set_weights(fl.get_weights())
            n_ok += 1
        except ValueError:
            print(f'  Skipped: {fl.name} -> {ql.name} (shape mismatch)')
    print(f'  Transferred weights for {n_ok} layer(s).')


# Custom-objects dict needed when reloading QKeras models
QKERAS_CUSTOM_OBJECTS = {}
_add_supported_quantized_objects(QKERAS_CUSTOM_OBJECTS)

#Import Training parameters needed for QAT

In [17]:
# ── Load Training parameters ─────────────────────────────────────────────
SPEC_PATH = os.path.join(DATASETS_PATH, "element_spec.pkl")
with open(SPEC_PATH, "rb") as f:
    specs = pickle.load(f)
    
MFCC_dataset_train_batches = tf.data.Dataset.load(
    os.path.join(DATASETS_PATH, "train_batches"),
    element_spec=specs["train"]
).prefetch(tf.data.AUTOTUNE)

MFCC_dataset_validation_batches = tf.data.Dataset.load(
    os.path.join(DATASETS_PATH, "validation_batches"),
    element_spec=specs["validation"]
).prefetch(tf.data.AUTOTUNE)

early_stopping_callback = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True
)
# ───────────────────────────────────────────────


# Compare configurations


In [ ]:

# QAT configs: (total_bits, integer_bits, fine_tune_epochs, learning_rate)
QAT_CONFIGS = [
    (16, 6, 30, 1e-5),
    ( 8, 4, 40, 5e-6),
    ( 4, 2, 60, 1e-6),
]

qat_models   = {}  # bits -> trained QKeras model
qat_results  = {}  # bits -> (loss, accuracy)

for bits, int_bits, ft_epochs, ft_lr in QAT_CONFIGS:
    label     = f'{bits}bit'
    ckpt_dir  = f'./ckpt_MFCC_QAT_{label}_{TARGET_SNR}'
    best_path = f'{ckpt_dir}/MFCC_QAT_{label}_best.keras'
    hist_path = f'{ckpt_dir}/MFCC_QAT_{label}_history.json'
    os.makedirs(ckpt_dir, exist_ok=True)

    print(f"\n{'='*60}")
    print(f'  QAT {bits}-bit  (integer_bits={int_bits})')
    print(f"{'='*60}")

    if os.path.exists(best_path):
        print(f'  Pre-trained QAT model found: {best_path}')
        print('  Loading.  Delete this file to retrain.')
        model_q = tf.keras.models.load_model(
            best_path, custom_objects=QKERAS_CUSTOM_OBJECTS
        )
        # FIX #7: always recompile with explicit parameters
        model_q.compile(
            optimizer=Adam(learning_rate=ft_lr),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )
    else:
        print(f"Building QAT model and transferring float weights ...")
        model_q = build_qat_model(MFCC_input_shape, NUM_CLASSES, bits, int_bits)
        model_q.compile(
            optimizer=Adam(learning_rate=ft_lr),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )
        transfer_float_weights(model_MFCC, model_q)

        # FIX #3: fresh callbacks per training session
        cbs = [
            tf.keras.callbacks.ModelCheckpoint(
                filepath=best_path, monitor='val_accuracy',
                save_best_only=True, verbose=1
            ),
            tf.keras.callbacks.EarlyStopping(
                monitor='val_loss', patience=max(10, PATIENCE // 2),
                restore_best_weights=True, verbose=1
            ),
        ]

        hist_q = model_q.fit(
            MFCC_dataset_train_batches,
            epochs=MAX_EPOCHS,
            validation_data=MFCC_dataset_validation_batches,
            callbacks=[early_stopping_callback]
        )
        save_history(hist_q, hist_path)

    if os.path.exists(hist_path):
        plot_training_history(load_history(hist_path),
                              title=f'QAT {bits}-bit - MFCC {TARGET_SNR}')

    qat_models[bits] = model_q


  QAT 16-bit  (integer_bits=6)
Building QAT model and transferring float weights ...


ValueError: Unrecognized keyword arguments: ['shape']

## Evaluate All Models

In [ ]:
all_results = []

loss, acc = evaluate_model(model_float, ds_test, unique_labels, title='Float 32-bit')
all_results.append({'model': 'Float 32-bit', 'accuracy': acc, 'loss': loss})

for bits in [16, 8, 4]:
    print(f'\n--- QAT {bits}-bit ---')
    loss, acc = evaluate_model(
        qat_models[bits], ds_test, unique_labels, title=f'QAT {bits}-bit'
    )
    all_results.append({'model': f'QAT {bits}-bit', 'accuracy': acc, 'loss': loss})

print('\n' + '='*50)
print(f'{"Model":<18} {"Accuracy":>10} {"Loss":>10}')
print('='*50)
for r in all_results:
    print(f"{r['model']:<18} {r['accuracy']:>10.4f} {r['loss']:>10.4f}")
print('='*50)

## hls4ml Synthesis – Vitis HLS → Bitstream-Ready IP

Each of the four models is converted to a Vitis HLS project and
synthesized for **`xck26-sfvc784-2LV-c`** at **200 MHz (5 ns)**.

The exported IP-XACT package can be added to a Vivado block design
to generate the final bitstream.


In [ ]:
def get_hls_config(keras_model, is_qat, bits=32):
    config = hls4ml.utils.config_from_keras_model(
        keras_model, granularity='name'
    )
    config['Model']['Strategy']    = 'Resource'  # minimize LUT / DSP
    config['Model']['ReuseFactor'] = 1

    if not is_qat:
        # Float model: use ap_fixed<32,12> for highest baseline accuracy
        config['Model']['Precision'] = 'ap_fixed<32,12>'
    # For QAT models hls4ml reads precision from QKeras quantizers automatically

    # The Flatten -> Dense(128) layer has 22 400 inputs.
    # xck26 has ~128 DSP48s. Minimum RF = ceil(22400/128) = 175.
    # Using 256 (next power-of-two) for alignment.
    LARGE_RF = 256
    for lname in config['LayerName']:
        if 'dense1' in lname:
            config['LayerName'][lname]['ReuseFactor'] = LARGE_RF
            config['LayerName'][lname]['Strategy']    = 'Resource'
    return config


def synthesize_model(keras_model, output_dir, label, is_qat=False, bits=32):
    print(f"\n{'─'*60}")
    print(f'  Synthesizing : {label}')
    print(f'  Output dir   : {output_dir}')
    print(f"{'─'*60}")

    config    = get_hls_config(keras_model, is_qat=is_qat, bits=bits)
    hls_model = hls4ml.converters.convert_from_keras_model(
        keras_model,
        hls_config   = config,
        output_dir   = output_dir,
        backend      = 'Vitis',
        part         = FPGA_PART,
        clock_period = CLOCK_PERIOD   # 5 ns -> 200 MHz
    )

    # C simulation checks functional correctness before RTL synthesis
    hls_model.compile()

    # Full synthesis + IP-XACT export (Vivado-importable bitstream-ready IP)
    hls_model.build(csim=False, synth=True, cosim=False, export=True)

    print(f'  Done: {label}')
    return hls_model


def parse_synth_report(output_dir, label):
    rpt = {'model': label}
    candidates = [
        os.path.join(output_dir, 'myproject_prj', 'solution1',
                     'syn', 'report', 'myproject_csynth.rpt'),
        os.path.join(output_dir, 'myproject_prj', 'solution1',
                     'syn', 'report', 'csynth.rpt'),
    ]
    rpt_path = next((p for p in candidates if os.path.exists(p)), None)
    if rpt_path is None:
        print(f'  Warning: report not found for {label}')
        return rpt
    content = open(rpt_path).read()

    # Latency
    m = re.search(
        r'Latency.*?Min.*?:(\s*)(\d+).*?Max.*?:(\s*)(\d+)',
        content, re.DOTALL | re.IGNORECASE
    )
    if m:
        rpt['lat_min_cyc'] = int(m.group(2))
        rpt['lat_max_cyc'] = int(m.group(4))
        rpt['lat_min_us']  = round(int(m.group(2)) * CLOCK_PERIOD / 1000, 3)
        rpt['lat_max_us']  = round(int(m.group(4)) * CLOCK_PERIOD / 1000, 3)

    # Resources
    for res in ['LUT', 'FF', 'DSP', 'BRAM_18K']:
        m = re.search(
            rf'\|?\s*{res}\s*\|?\s*(\d+)\s*\|?\s*(\d+)\s*\|?\s*([\d.]+)\s*%',
            content
        )
        if m:
            rpt[f'{res}_used'] = int(m.group(1))
            rpt[f'{res}_avail']= int(m.group(2))
            rpt[f'{res}_pct']  = float(m.group(3))
    return rpt

## Run Synthesis

In [ ]:
os.makedirs(HLS_BASE_DIR, exist_ok=True)

synth_targets = [
    # (model,           subdir,      label,         is_qat, bits)
    (model_float,       'float_32',  'Float 32-bit', False, 32),
    (qat_models[16],    'qat_16bit', 'QAT 16-bit',   True,  16),
    (qat_models[8],     'qat_8bit',  'QAT 8-bit',    True,   8),
    (qat_models[4],     'qat_4bit',  'QAT 4-bit',    True,   4),
]

hls_handles   = {}
synth_reports = []

for model, subdir, label, is_qat, bits in synth_targets:
    out_dir = os.path.join(HLS_BASE_DIR, subdir)
    hm = synthesize_model(model, out_dir, label, is_qat=is_qat, bits=bits)
    hls_handles[label]   = hm
    synth_reports.append(parse_synth_report(out_dir, label))

## Results Summary

In [ ]:
# ── Accuracy / loss ──────────────────────────────────────────────────────
print('='*55)
print(f'{"Model":<18} {"Accuracy":>10} {"Loss":>10}')
print('='*55)
for r in all_results:
    print(f"{r['model']:<18} {r['accuracy']:>10.4f} {r['loss']:>10.4f}")
print('='*55)

# ── Synthesis resources & latency ────────────────────────────────────────
print('\n')
df_rpt = pd.DataFrame(synth_reports)
want   = ['model',
          'lat_min_us', 'lat_max_us',
          'LUT_used', 'LUT_pct',
          'FF_used',  'FF_pct',
          'DSP_used', 'DSP_pct',
          'BRAM_18K_used', 'BRAM_18K_pct']
show   = [c for c in want if c in df_rpt.columns]
print(df_rpt[show].to_string(index=False))

print(f'\nAll models targeted at {FPGA_PART} @ {1000//CLOCK_PERIOD} MHz')
print('IP-XACT packages are in <output_dir>/myproject_prj/solution1/impl/ip/')
print('Import them into a Vivado block design and run Generate Bitstream.')